# Edge-Weighting Ablation Study Analysis

Statistical analysis of 140 configurations testing edge-weighted optimization for VRA compliance.

**Configurations**: 7 weight factors × 4 thresholds × 5 states = 140 tests

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression, LinearRegression

# Load data
df = pd.read_csv('../results/edge_weighting_ablation_study.csv')
print(f"Loaded {len(df)} configurations")
df.head()

## 1. Success Rate Analysis

In [ ]:
# Overall success rate
overall_success = df['success'].mean()
print(f"Overall Success Rate: {overall_success:.1%}")

# By state
state_success = df.groupby('state')['success'].agg(['mean', 'sum', 'count'])
state_success.columns = ['Success Rate', 'Successes', 'Total']
print("\nSuccess Rate by State:")
print(state_success.sort_values('Success Rate', ascending=False))

## 2. Parameter Sensitivity

In [ ]:
# Weight factor sensitivity
weight_success = df.groupby('weight_factor')['success'].agg(['mean', 'count'])
weight_success.columns = ['Success Rate', 'Configurations']
print("Success Rate by Weight Factor:")
print(weight_success)

# Threshold sensitivity
threshold_success = df.groupby('minority_threshold')['success'].agg(['mean', 'count'])
threshold_success.columns = ['Success Rate', 'Configurations']
print("\nSuccess Rate by Minority Threshold:")
print(threshold_success)

## 3. Compactness Impact Analysis

In [ ]:
# Compare edge cut: success vs fail
success_df = df[df['success'] == True]
fail_df = df[df['success'] == False]

print(f"Success configs - Mean edge cut: {success_df['edge_cut_unweighted'].mean():.1f}")
print(f"Fail configs - Mean edge cut: {fail_df['edge_cut_unweighted'].mean():.1f}")

# Statistical test
t_stat, p_value = stats.ttest_ind(success_df['edge_cut_unweighted'], 
                                   fail_df['edge_cut_unweighted'])
print(f"\nT-test: t={t_stat:.3f}, p={p_value:.4f}")

# By state: baseline vs optimal
print("\nCompactness Change (Baseline to Optimal Success):")
for state in df['state'].unique():
    state_df = df[df['state'] == state]
    baseline = state_df[state_df['weight_factor'] == 1].iloc[0]
    success_configs = state_df[state_df['success'] == True]
    
    if len(success_configs) > 0:
        optimal = success_configs.loc[success_configs['edge_cut_unweighted'].idxmin()]
        change = optimal['edge_cut_unweighted'] - baseline['edge_cut_unweighted']
        pct_change = change / baseline['edge_cut_unweighted'] * 100
        print(f"{state}: {baseline['edge_cut_unweighted']:.0f} → {optimal['edge_cut_unweighted']:.0f} "
              f"({change:+.0f}, {pct_change:+.1f}%)")

## 4. Predictive Modeling

In [ ]:
# Logistic regression: What predicts success?
# Features: weight factor (log), threshold, state minority %

# Add state minority percentage
state_minority = {'AL': 0.369, 'GA': 0.424, 'LA': 0.416, 'MS': 0.461, 'SC': 0.351}
df['state_minority_pct'] = df['state'].map(state_minority)

# Prepare features
X = df[['weight_factor', 'minority_threshold', 'state_minority_pct']].copy()
X['log_weight'] = np.log10(X['weight_factor'] + 1)
X = X[['log_weight', 'minority_threshold', 'state_minority_pct']]
y = df['success'].astype(int)

# Fit model
lr = LogisticRegression()
lr.fit(X, y)

print("Logistic Regression Coefficients:")
for feature, coef in zip(X.columns, lr.coef_[0]):
    print(f"  {feature}: {coef:.3f}")
print(f"\nModel accuracy: {lr.score(X, y):.1%}")

In [ ]:
# Linear regression: What predicts maximum minority %?
X_reg = df[['weight_factor', 'minority_threshold', 'state_minority_pct']].copy()
X_reg['log_weight'] = np.log10(X_reg['weight_factor'] + 1)
X_reg = X_reg[['log_weight', 'minority_threshold', 'state_minority_pct']]
y_reg = df['max_minority_pct']

reg = LinearRegression()
reg.fit(X_reg, y_reg)

print("Linear Regression Coefficients (Max Minority %):")
for feature, coef in zip(X_reg.columns, reg.coef_):
    print(f"  {feature}: {coef:.4f}")
print(f"\nR² score: {reg.score(X_reg, y_reg):.3f}")

## 5. Optimal Configuration Identification

In [ ]:
# For each state, find optimal config (lowest weight factor achieving success)
print("Optimal Configuration per State:")
print("="*70)

for state in ['MS', 'GA', 'LA', 'AL', 'SC']:
    state_df = df[df['state'] == state]
    success_df = state_df[state_df['success'] == True]
    
    if len(success_df) > 0:
        optimal = success_df.loc[success_df['weight_factor'].idxmin()]
        print(f"{state}: {optimal['weight_factor']}x @ {optimal['minority_threshold']*100:.0f}% "
              f"→ {optimal['mm_count']}/{optimal['target_mm']} MM "
              f"({optimal['max_minority_pct']*100:.1f}% max, "
              f"edge cut: {optimal['edge_cut_unweighted']:.0f})")
    else:
        best_effort = state_df.loc[state_df['max_minority_pct'].idxmax()]
        print(f"{state}: NO SUCCESS - Best: {best_effort['weight_factor']}x @ "
              f"{best_effort['minority_threshold']*100:.0f}% "
              f"→ {best_effort['max_minority_pct']*100:.1f}% max "
              f"({50 - best_effort['max_minority_pct']*100:.1f} points short)")

## 6. Visualization: Success Probability Heatmap

In [ ]:
# Create heatmap: weight factor × threshold, showing success rate
pivot = df.pivot_table(
    index='minority_threshold',
    columns='weight_factor',
    values='success',
    aggfunc='mean'
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn', center=0.5,
            linewidths=0.5, cbar_kws={'label': 'Success Rate'})
plt.title('VRA Success Rate by Weight Factor and Threshold\n(Averaged Across All 5 States)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Weight Factor', fontweight='bold')
plt.ylabel('Minority Threshold', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/success_probability_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Summary Table for Paper

In [ ]:
# Generate LaTeX table
summary = []
for state in ['MS', 'GA', 'LA', 'AL', 'SC']:
    state_df = df[df['state'] == state]
    success_configs = state_df[state_df['success'] == True]
    
    row = {
        'State': state,
        'State Minority %': state_minority[state] * 100,
        'Success Rate': state_df['success'].mean() * 100,
        'Best MM Count': state_df['mm_count'].max(),
        'Target MM': state_df['target_mm'].iloc[0],
        'Best Max %': state_df['max_minority_pct'].max() * 100
    }
    
    if len(success_configs) > 0:
        optimal = success_configs.loc[success_configs['weight_factor'].idxmin()]
        row['Optimal Config'] = f"{optimal['weight_factor']}x @ {optimal['minority_threshold']*100:.0f}%"
    else:
        row['Optimal Config'] = 'None'
    
    summary.append(row)

summary_df = pd.DataFrame(summary)
print("\nSummary Table:")
print(summary_df.to_string(index=False))

# Export to LaTeX
latex = summary_df.to_latex(index=False, float_format='%.1f')
print("\nLaTeX Table:")
print(latex)

## 8. Key Findings

### Success Rates
- Overall: 80% of states achieve VRA compliance (4/5)
- By state: MS/GA 100%, LA ~50%, AL ~40%, SC 0%

### Parameter Sensitivity
- Weight factors 5x-100x perform best
- Lower thresholds (40-45%) achieve higher success
- Very high weights (500x-1000x) cause over-clustering

### Compactness Impact
- Average edge cut increase: +4%
- Alabama actually improves compactness (-1.4%)
- No significant difference between success and fail configs

### Optimal Configurations
- MS: 1x @ 40% (baseline sufficient)
- GA: 1x @ 40% (baseline sufficient)
- LA: 5x @ 40% (minimal weighting needed)
- AL: 5x @ 40% (breakthrough at low weight)
- SC: None (geographic limit at 35.1% minority)